# 📋 Финальный анализ спроса с учётом всех критериев из ТЗ

In [1]:
!pip install -q streamlit pyngrok
!apt install -y unzip
!wget -q -c https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip -O ngrok.zip
!unzip -o ngrok.zip
!mv ngrok /usr/local/bin


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Archive:  ngrok.zip
  inflating: ngrok                   


In [2]:
from pyngrok import conf, ngrok
conf.get_default().auth_token = "2uRWQsVBhRrursScvzOfGGB5gYq_4pAq9D9vD8RKjvPfmvriY"


In [3]:
!pip install -q streamlit pyngrok

# Сохраняем адаптированное приложение
app_code = '''
import pandas as pd
import numpy as np
import streamlit as st
import matplotlib.pyplot as plt
import sqlite3
from sklearn.linear_model import LinearRegression

st.set_page_config(layout="wide")
st.title("📊 Финальный дашборд оценки спроса и закупок")

# 📁 Подключаем заранее загруженный файл базы данных
db_path = "demand_forecast.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM forecast_data", conn)
conn.close()

df["Дата"] = pd.to_datetime(df["Дата"])
df["Квартал"] = df["Дата"].dt.to_period("Q")

df = df.drop(columns=["Срок_изготовления_дней"], errors="ignore")
df["Остатки_на_складе"] *= 1000
df["Поставки_в_единицах"] *= 1000
df["Маркетинговая_активность"] = df["Дата"].dt.month.apply(lambda m: 1 if m in [3,4,5,9,10,11] else 0)

кварталы = sorted(df["Квартал"].astype(str).unique())
st.sidebar.header("⚙️ Настройка фильтров")
выбранный_квартал = st.sidebar.selectbox("Выберите квартал:", кварталы)
min_остатки, max_остатки = st.sidebar.slider("Остатки на складе (тыс.)", 0, int(df["Остатки_на_складе"].max()), (0, 300000))
маркетинг = st.sidebar.selectbox("Маркетинговая активность:", ["Любая", "Да", "Нет"])

df_filtered = df[df["Квартал"].astype(str) == выбранный_квартал]
df_filtered = df_filtered[(df_filtered["Остатки_на_складе"] >= min_остатки) & (df_filtered["Остатки_на_складе"] <= max_остатки)]

if маркетинг == "Да":
    df_filtered = df_filtered[df_filtered["Маркетинговая_активность"] == 1]
elif маркетинг == "Нет":
    df_filtered = df_filtered[df_filtered["Маркетинговая_активность"] == 0]

st.subheader("📋 Отфильтрованные данные")
st.dataframe(df_filtered)

st.subheader("📦 Необходимые закупки (пример)")
example_materials = {
    "Пружинный блок": 1200,
    "Пенополиуретан": 800,
    "Клей": 400,
    "Латекс натуральный": 250,
    "Латекс искусственный": 380,
    "Кокосовая койра": 320,
    "Мемори-фоам": 150
}
st.table(pd.DataFrame.from_dict(example_materials, orient="index", columns=["Необходимо закупить (ед.)"]))

col1, col2 = st.columns(2)

with col1:
    st.markdown("### 📦 Остатки и поставки")
    fig1, ax1 = plt.subplots()
    ax1.bar(df_filtered["Дата"].dt.strftime("%Y-%m-%d"), df_filtered["Остатки_на_складе"], label="Остатки", alpha=0.6)
    ax1.bar(df_filtered["Дата"].dt.strftime("%Y-%m-%d"), df_filtered["Поставки_в_единицах"], label="Поставки", alpha=0.6)
    ax1.set_xticklabels(df_filtered["Дата"].dt.strftime("%Y-%m-%d"), rotation=45)
    ax1.legend()
    st.pyplot(fig1)

    st.markdown("### 📈 Маркетинговая активность")
    marketing_counts = df_filtered["Маркетинговая_активность"].value_counts()
    labels = ["Есть активность" if i==1 else "Нет активности" for i in marketing_counts.index]
    fig2, ax2 = plt.subplots()
    ax2.pie(marketing_counts, labels=labels, autopct="%1.1f%%", startangle=90)
    st.pyplot(fig2)

with col2:
    st.markdown("### 🔮 Прогноз спроса")
    if "Спрос_на_закупки" in df.columns:
        X = df_filtered.drop(columns=["Дата", "Квартал", "Спрос_на_закупки"])
        y = df_filtered["Спрос_на_закупки"]
        if not X.empty:
            model = LinearRegression().fit(X, y)
            inputs = {col: st.number_input(f"{col}", value=float(df[col].mean())) for col in X.columns}
            input_df = pd.DataFrame([inputs])

            if st.button("Сделать прогноз"):
                result = model.predict(input_df)[0]
                st.success(f"🟢 Прогноз спроса: {result:.2f} единиц")
                if result > df["Спрос_на_закупки"].mean():
                    st.info("Рекомендация: увеличить объёмы закупок.")
                else:
                    st.warning("Рекомендация: спрос ниже среднего — действовать сдержанно.")
'''





In [4]:
# Сохраняем Streamlit-приложение в файл
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

# ⏳ Импортируем модули
import threading
import time
import os
import socket
from pyngrok import ngrok

# ✅ Запуск Streamlit в фоновом потоке
def run():
    os.system("streamlit run app.py")

thread = threading.Thread(target=run)
thread.start()

# ✅ Ожидание запуска Streamlit на порту 8501
def wait_for_port(port, host="127.0.0.1", timeout=120):
    print("⏳ Ожидаем запуск Streamlit...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            with socket.create_connection((host, port), timeout=2):
                print("✅ Streamlit успешно запущен!")
                return True
        except OSError:
            time.sleep(2)
    print("❌ Streamlit не запустился за 2 минуты.")
    return False

# ✅ Подключение ngrok после запуска
if wait_for_port(8501):
    public_url = ngrok.connect(8501)
    print("🚀 Откройте приложение по ссылке:", public_url)


⏳ Ожидаем запуск Streamlit...
✅ Streamlit успешно запущен!
🚀 Откройте приложение по ссылке: NgrokTunnel: "https://8155-35-194-129-208.ngrok-free.app" -> "http://localhost:8501"
